In [5]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt
import time
from sklearn.preprocessing import StandardScaler
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

Device: cuda


In [ ]:
stap_per_dag = 24 * 12
# aantal dagen lengte van de imput    
aantal_dagen_imput = 2             
imput_len = aantal_dagen_imput * stap_per_dag
# aantal dagen van de voorspelling
aantal_dagen_forecast = 3
horizon= aantal_dagen_forecast * stap_per_dag
stride = 12

TARGET_COL= "POW"
FEATURE_COLS= []  # alle features ook de target

HIDDEN_SIZE   = 64
NUM_LAYERS    = 2
DROPOUT       = 0.2
BATCH_SIZE    = 32
EPOCHS        = 30
LR            = 1e-3
PATIENCE      = 5

TRAIN_FRAC = 0.70 # percentage als train data
VAL_FRAC = 0.15  # percentage als validatie data

## DATA LADEN 
### eisen: opgeschoond en geen nul waarden



In [ ]:
df = 

## Splitten van de data CHRONOLOGISCH


In [ ]:
TARGET_IDX = FEATURE_COLS.INDEX(TARGET_COL)
N_FEATURES= len(FEATURE_COLS)

n=  len(df)
i_train, i_val = int(n * TRAIN_FRAC), int(n * (TRAIN_FRAC + VAL_FRAC))
values= df[FEATURE_COLS].values
# train test split op chronologische volgored
train_raw, val_raw, test_raw = values[:i_train], values[i_train:i_val], values[i_val:]

# scalen van de data (de scler word gefit op de TRAIN data!!!!!)
scaler = StandardScaler().fit(train_raw)
train = scaler.transform(train_raw).astype(np.float32)
val   = scaler.transform(val_raw).astype(np.float32)
test  = scaler.transform(test_raw).astype(np.float32)

NameError: name 'FEATURE_COLS' is not defined

# dataset (sliding window)

In [7]:
class WindowDataset(Dataset):

    def __init__(self, data, input_len, horizon, target_idx, stride=1):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.input_len, self.horizon, self.target_idx = input_len, horizon, target_idx
        n = len(data) - input_len - horizon + 1
        self.starts = range(0, max(n, 0), stride)

    def __len__(self):
        return len(self.starts)

    def __getitem__(self, i):
        s = self.starts[i]
        x = self.data[s : s + self.input_len]
        y = self.data[s + self.input_len : s + self.input_len + self.horizon, self.target_idx]
        return x, y

train_dl = DataLoader(WindowDataset(train, imput_len, horizon, TARGET_IDX, stride), batch_size=BATCH_SIZE, shuffle=True)
val_dl   = DataLoader(WindowDataset(val,   imput_len, horizon, TARGET_IDX, stride), batch_size=BATCH_SIZE)
test_dl  = DataLoader(WindowDataset(test,  imput_len, horizon, TARGET_IDX, stride), batch_size=BATCH_SIZE)


NameError: name 'train' is not defined

# LSTM model


In [ ]:
class LSTM(nn.Module):
    def __init__ (self, n_features= N_FEATURES, hidden_size= HIDDEN_SIZE, num_layers = NUM_LAYERS, dropout= DROPOUT)
        super.__init__()
        self.rnn= nn.LSTM(input_size=n_features, hidden_size=hidden_size,
                          num_layers=num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.head = nn.Linear(hidden_size, horizon)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.head(out[:, -1, :])

In [ ]:
@torch.no_grad()
def evaluate_loss(model, loader, loss_fn):
    # bereken van de loss functie 
    model.eval()                                    # zet het model in evaluatiemodus 
    total = 0.0
    for x, y in loader:                             # loop over alle batches
        x, y = x.to(DEVICE), y.to(DEVICE)           
        total += loss_fn(model(x), y).item() * len(x)
    return total / len(loader.dataset)              # gemiddelde fout over alle voorbeelden


def train_model(model, train_loader, val_loader, epochs=EPOCHS, lr=LR, patience=PATIENCE):
    """Traint het model en geeft het beste model + het verloop van de fouten terug."""
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)   # optimizer: past de gewichten aan
    loss_fn = nn.L1Loss()                              # mean absolute error

    # Bijhouden voor early stopping
    best_val = float("inf")     
    bad = 0                     
    best_state = None           
    history = {"train": [], "val": []}   # voor de plots 

    for epoch in range(epochs):
   
        model.train()                                   
        train_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()                             # gradient resetten
            loss = loss_fn(model(x), y)                 # 2. voorspellen + fout berekenen
            loss.backward()                             # 3. back propegation
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)  
            opt.step()                                  #  gewichten aanpassen
            train_loss += loss.item() * len(x)
        train_loss /= len(train_loader.dataset)         # gemiddelde loss

        val_loss = evaluate_loss(model, val_loader, loss_fn)   # fout op ongeziene data

        history["train"].append(train_loss)
        history["val"].append(val_loss)
        print(f"epoch {epoch+1:02d} | train {train_loss:.4f} | val {val_loss:.4f}")
